In [ ]:
# clone DECODE repository
import os
import subprocess

if not os.path.exists("../decode"):
    subprocess.run(["git", "clone", "https://github.com/waldnerf/decode.git", "../decode"])
    

In [ ]:
# download pretrained model from airbus france india dataset (Wang et al. 2022)

import urllib.request
os.makedirs("../model", exist_ok=True)
if not os.path.exists("../model/airbus_france_india.params"):
    urllib.request.urlretrieve(
        "https://zenodo.org/records/7315090/files/india_Airbus_SPOT_model.params?download=1",
        "../model/airbus_france_india.params")

In [ ]:
import sys
sys.path.append('/data/Aldhani/cv_fields/code/decode/')
sys.path.append('/data/Aldhani/cv_fields/code/')
sys.path.append('/data/Aldhani/cv_fields/code/smallholder-fields/')

In [ ]:
import glob
import time
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
from osgeo import gdal
from osgeo import ogr

import mxnet as mx
from mxnet import nd, gpu, gluon, autograd, npx, image
from mxnet.gluon.data import DataLoader
from mxnet.gluon.data.vision import transforms
from mxnet.gluon.loss import Loss

In [ ]:
from decode.FracTAL_ResUNet.nn.loss.mtsk_loss import *
from decode.FracTAL_ResUNet.models.heads.head_cmtsk import *
from decode.FracTAL_ResUNet.models.semanticsegmentation.FracTAL_ResUNet_features import *
from decode.FracTAL_ResUNet.models.semanticsegmentation.FracTAL_ResUNet import FracTAL_ResUNet_cmtsk

from helpers_model import *

In [ ]:
# image and label paths
# file naming must align between images and labels
# current naming convention is {country}_tile_{epsg}_{tile_x}_{tile_y}_{yyyymmdd}.tif for images and 
# {country}_tile_{epsg}_{tile_x}_{tile_y}_{yyyymmdd}_mtsk.tif for labels
# for example: mozambique_tile_32636_-0019_-0279_20230503.tif

img_folder = '../sample_data/'
lbl_folder = '../labels/'

In [ ]:
# create chips of 256x256 pixels for training
image_names = glob.glob(img_folder+'*.tif')
label_names = glob.glob(lbl_folder+'*_mtsk.tif')

print(len(label_names))
print(len(image_names))

if not os.path.exists(img_folder+'/chips'): os.makedirs(img_folder+'/chips')
if not os.path.exists(lbl_folder+'/chips'): os.makedirs(lbl_folder+'/chips')

start = 0
step = 256
n=0

for n, _ in enumerate(image_names):
  image = image_names[n]
  label = label_names[n]

  ds = gdal.Open(image_names[n])
  width = ds.RasterXSize
  height = ds.RasterYSize
  print(width)
  print(height)
  x_count = int(np.floor(width/step))
  y_count = int(np.floor(height/step))

  for x_step in range(0, x_count):
        for y_step in range(0, y_count):
            window = (x_step*step, y_step*step, step, step)
            out_image = f'{img_folder}/chips/{os.path.basename(image)[:-4]}_{x_step:02d}_{y_step:02d}.tif'
            out_label = f'{lbl_folder}/chips/{os.path.basename(label)[:-4]}_{x_step:02d}_{y_step:02d}.tif'
            if not os.path.exists(out_image):
                  gdal.Translate(out_image, image, srcWin = window, creationOptions=['COMPRESS=LZW'])
            if not os.path.exists(out_label):
                  gdal.Translate(out_label, label, srcWin = window, creationOptions=['COMPRESS=LZW'])


In [ ]:
# adjust image and label paths to folder containing chips
img_folder = '../sample_data/chips'
lbl_folder = '../labels/chips'

In [ ]:
# random train val test split

# get tile ids from labels
tile_ids = sorted(glob.glob(lbl_folder + '/*.tif'))
print(len(tile_ids))

# create substrings for image and label paths,
# needs adjusting if file naming convention changes
image_names = sorted([f"{img_folder}/{os.path.basename(tile_id)[:-15]}_{os.path.basename(tile_id)[-9:]}" for tile_id in tile_ids])
label_names = sorted([f"{lbl_folder}/{os.path.basename(tile_id)}" for tile_id in tile_ids])

# print examples / length of images & labels
m = [print(image_names[i], label_names[i]) for i in range(0,3)]
print(len(tile_ids), len(image_names), len(label_names))

# define split 
probs=[0.60, 0.20, 0.20]

### randomly select w leakage across tiles
np.random.seed(42)
split = np.random.choice(3, len(tile_ids), replace=True, p=probs)
train_image_names = np.array(image_names)[split==0]
val_image_names = np.array(image_names)[split==1]
test_image_names = np.array(image_names)[split==2]

train_label_names = np.array(label_names)[split==0]
val_label_names = np.array(label_names)[split==1]
test_label_names = np.array(label_names)[split==2]

print('N train, val, test')
print(len(train_image_names), len(train_label_names), len(val_image_names), len(val_label_names), len(test_image_names), len(test_label_names))


In [ ]:
##############################################
# training params
pretrained_model = '../model/airbus_france_india.params'

model_type = 'fractal-resunet'

epochs = 10
lr = 0.001 
lr_decay = None
n_filters = 32
depth = 6
n_classes = 1
batch_size = 12

month = 'spot'
bands='rgb'
train_val='60-20'
lossf='ftnmt_masked'
folder_suffix = 'finetune-v01'

ctx_name = 'gpu'
gpu_id = 0

In [ ]:
# execute training run
model = run(train_image_names, val_image_names,
    train_label_names, val_label_names,
    trained_model=pretrained_model,
    epochs=epochs, lr=lr, lr_decay=lr_decay,
    model_type=model_type, n_filters=n_filters, depth=depth, n_classes=n_classes,
    batch_size=batch_size, month=month, bands=bands,
    train_val=train_val, normalize='none', maskedloss=True, otf_aug=True, lossf=lossf,
    ctx_name=ctx_name,
    gpu_id=gpu_id,
    folder_suffix=folder_suffix)